In [20]:
import requests
import json
import pandas as pd
from datetime import datetime
from keys import flights_key  # tu API key de AerodataBox
import sqlalchemy
import pymysql

In [21]:
airport_icao = "MMMX"   # Aeropuerto Internacional de Ciudad de México
date_from = "2025-09-01T08:00"
date_to   = "2025-09-01T20:00"

url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport_icao}/{date_from}/{date_to}"

querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "true",
    "withPrivate": "true",
    "withLocation": "false"
}

headers = {
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "x-rapidapi-key": flights_key
}

existing_airports = pd.read_sql("SELECT icao_code FROM airport", engine)
flights_df = flights_df[flights_df['icao_code'].isin(existing_airports['icao_code'])]



In [22]:
response = requests.get(url, headers=headers, params=querystring)

if response.status_code != 200:
    print("❌ Error al obtener datos:", response.status_code)
    flights_data = []
else:
    flights_data = response.json().get("arrivals", [])
    print("✅ Datos obtenidos correctamente:", len(flights_data), "vuelos encontrados.")


✅ Datos obtenidos correctamente: 263 vuelos encontrados.


In [23]:
def extract_flight(flight):
    # Convertir fecha a formato compatible con MySQL
    scheduled_time = flight["arrival"]["scheduledTime"]["local"]
    scheduled_time = pd.to_datetime(scheduled_time).strftime("%Y-%m-%d %H:%M:%S")

    icao_code = flight["departure"]["airport"].get("icao", None)
    
    return {
        "scheduled_arrival_time": scheduled_time,
        "flight_number": flight.get("number", None),
        "from_airport": flight["departure"]["airport"].get("name", None),
        "airline": flight["airline"].get("name", None),
        "aircraft": flight["aircraft"].get("model", None),
        "icao_code": icao_code
    }


In [24]:
flights_list = [extract_flight(f) for f in flights_data]
flights_df = pd.DataFrame(flights_list)

print("Vista previa de los datos obtenidos:")
display(flights_df.head())


Vista previa de los datos obtenidos:


,scheduled_arrival_time,flight_number,from_airport,airline,aircraft,icao_code
0,2025-09-01 08:35:00,UA 1024,Houston,United,Boeing 737-700,KIAH
1,2025-09-01 08:30:00,AM 905,Monterrey,Aeromexico,Boeing 737-900,MMMY
2,2025-09-01 08:30:00,AM 505,Cancún,Aeromexico,Boeing 737-800,MMUN
3,2025-09-01 08:35:00,AM 205,Guadalajara,Aeromexico,Boeing 737,MMGL
4,2025-09-01 08:40:00,VB 1355,Monterrey,VivaAerobus,Airbus A321 NEO,MMMY


In [25]:
schema = "gans"
host = "127.0.0.1"
user = "root"
password = "Misterio619"
port = 3306

engine = sqlalchemy.create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
)

In [26]:
try:
    flights_df.to_sql("flight_arrival", con=engine, if_exists="append", index=False)
    print("✅ Datos insertados correctamente en la tabla 'flight_arrival'")
except Exception as e:
    print("❌ Error al insertar datos:", e)

✅ Datos insertados correctamente en la tabla 'flight_arrival'
